# Area of Interest (AOI) Delineation for Nigerian Cities

## What This Notebook Does

This notebook **delineates urban Areas of Interest (AOI)** for major Nigerian cities using a data-driven approach that combines population data, urban extent polygons, and administrative boundaries. Each city receives a validated AOI boundary used as the spatial foundation for all subsequent LST analyses.

---

## Table of Contents

Navigate directly to any section:

1. [Setup & Imports](#setup)
2. [Load Required Datasets](#load-data)
3. [City Selection](#city-selection)
4. [Process City AOIs](#process-aoi)
5. [Export Data](#export-data)
6. [Visualizations](#visualizations)
   - [6a. Per-City Maps & Population Trends](#viz-per-city)
   - [6b. Composite Population Trend](#viz-composite)
7. [Summary & Next Steps](#summary)

---

## Sub-Research Objectives

1. **Define reproducible urban boundaries** for 67 Nigerian cities (≥300k, WUP 2025) using standardized geospatial criteria
2. **Quantify AOI metrics** including area (km²), population coverage, and urban extent efficiency
3. **Identify co-located cities** within buffer zones for regional urban heat island context
4. **Export standardized GeoJSON boundaries** for use in LST extraction (Notebooks 03, 04) and hotspot analysis (Notebook 05)

---

## Data Sources

| Dataset | Source | Purpose |
|---------|--------|---------|
| **Cities CSV** | UN World Urbanization Prospects (WUP 2025) | City coordinates and population estimates (1975–2050) |
| **Urban Extent GeoJSON** | Urban Landscape Character / GHSL | Urban agglomeration polygons for boundary delineation |
| **Nigeria Boundary GeoJSON** | GADM / Natural Earth | National boundary for spatial clipping and context |

---

## Notebook Structure

### Processing Pipeline

```
┌┐-------------------------------------------------------------
│ SECTION 1: Setup & Imports                                  │
│ • Load libraries and project configuration                  │
│ • Set buffer distance from config.py                        │
│ • Initialize paths and parameters                           │
└──┐-----------------------------------------------------------˜
                            ↓
┌┐-------------------------------------------------------------
│ SECTION 2: Load Required Datasets                           │
│ • Load cities CSV (coordinates + population)                │
│ • Load urban extent GeoJSON (urban polygons)                │
│ • Load Nigeria boundary GeoJSON                             │
│ • Validate data integrity and CRS                           │
└──┐-----------------------------------------------------------˜
                            ↓
┌┐-------------------------------------------------------------
│ SECTION 3: City Selection                                   │
│ • Read selection mode from config.py                        │
│ • Options: ALL (791), ABOVE_1M (17), ABOVE_300K (67), TEST  │
│ • Display selected cities and available modes               │
└──┐-----------------------------------------------------------˜
                            ↓
┌┐-------------------------------------------------------------
│ SECTION 4: Process City AOIs                                │
│ • Buffer urban polygons by configured distance              │
│ • Compute AOI geometry, area, and coverage metrics          │
│ • Identify neighbouring cities within AOI bounds            │
│ • Store results in dictionary for export                    │
└──┐-----------------------------------------------------------˜
                            ↓
┌┐-------------------------------------------------------------
│ SECTION 5: Export Data                                      │
│ • Export AOI GeoJSON boundaries per city                    │
│ • Export CSV population/metrics tables                      │
│ • Export comprehensive statistics JSON                      │
│ • Organize into City_[Name]/ folder structure               │
└──┐-----------------------------------------------------------˜
                            ↓
┌┐-------------------------------------------------------------
│ SECTION 6: Visualizations                                   │
│ • 6a. Per-city: static AOI map, interactive map, pop trend  │
│ • 6b. Composite population trend (all cities, 1950–2035)    │
│ • Save figures as SVG + PDF                                 │
└──┐-----------------------------------------------------------˜
```

### Code Organization

All heavy processing is **modularized into reusable utilities** (`src/` directory):

- **`aoi_utils.py`**: AOI delineation, buffer operations, urban polygon matching
- **`data_io.py`**: Standardized CSV/GeoJSON/JSON exports with consistent naming
- **`visualization.py`**: AOI maps, interactive maps, population trend plots, composite population trend

## Output Structure

All results are saved to: `data/outputs/01_aoi_delineation/City_{city_name}/`

```
City_Lagos/
├── Features/
│   ├── Lagos_AOI_Boundary.geojson        # Full AOI boundary
│   ├── Lagos_Main_Polygon.geojson        # Core urban polygon
│   └── Lagos_Adjacent_Polygons.geojson   # Neighbouring urban polygons
├── Data/
│   ├── Lagos_Cities_In_AOI.csv           # Cities within buffer zone
│   └── Lagos_Comprehensive_Stats.json    # Area, population, metrics
└── Figures/
    ├── Lagos_AOI_Map.svg             # Static AOI map
    ├── Lagos_Interactive_Map.html        # Interactive folium map
    └── Lagos_Population_Trends.svg   # Population projection plot

Composite_Population_Trend.svg        # Aggregated trend (root level)
```

---

## Configuration

Analysis parameters are centralized in **`config.py`**:

```python
CITY_SELECTION_MODE = "ALL"   # "ALL", "ABOVE_1M", "TEST", "SPECIFIC"
AOI_BUFFER_DISTANCE = 5000    # Buffer in metres (5 km)
```

Modify `config.py` to adjust city selection and buffer distance without changing notebook code.

---

<a id="setup"></a>
## 1. Setup & Imports

Load project libraries, configuration, and define analysis parameters.

**What This Section Does**:
- Imports core Python packages and project utilities from `src/`
- Sets buffer distance from `config.py`
- Initializes Papermill parameters for automated execution

**Expected Output**: Confirmation of successful import and configuration values

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
"""
AOI Protocol for Nigerian Major Cities - Modularized Version
"""

# Core imports
import sys
from pathlib import Path

# Add project root and src to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

# Import configuration and utilities
import config
from aoi_utils import *
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from IPython.display import display

# Papermill parameters (for automated execution)
buffer_distance = config.AOI_BUFFER_DISTANCE
output_enabled = True

print("All modules imported successfully!")
print(f"Project root: {config.PROJECT_ROOT}")
print(f"Using buffer distance: {buffer_distance/1000}km")

<a id="load-data"></a>
## 2. Load Required Datasets

**Purpose**: Load the three key geospatial datasets needed for AOI delineation.

**What This Section Does**:
- Loads cities CSV with coordinates and population estimates (UN World Urbanization Prospects (WUP 2025))
- Loads urban extent polygons (GHS-UCDB / GHSL GeoJSON)
- Loads Nigeria national boundary (GADM / Natural Earth GeoJSON)
- Validates CRS compatibility and data integrity

**Expected Output**: Three DataFrames/GeoDataFrames ready for spatial processing

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# Load all required datasets using centralized function
try:
    cities_df, urban_gdf, boundary_gdf = load_aoi_datasets(
        str(config.get_aoi_data_path('cities_csv')),
        str(config.get_aoi_data_path('urban_geojson')),
        str(config.get_aoi_data_path('boundary_geojson'))
    )
    
    # Validate configuration
    config.validate_config()
    
    print(f"Data loaded successfully!")
    print(f"Available cities: {len(cities_df)}")
    print(f"Urban polygons: {len(urban_gdf)}")
    print(f"Boundary CRS: {boundary_gdf.crs}")
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please check your data file paths in config.py")
    
    # Debug information
    print(f"Expected cities CSV: {config.get_aoi_data_path('cities_csv')}")
    print(f"Expected urban GeoJSON: {config.get_aoi_data_path('urban_geojson')}")
    print(f"Expected boundary GeoJSON: {config.get_aoi_data_path('boundary_geojson')}")

<a id="city-selection"></a>
## 3. City Selection

**Purpose**: Determine which cities to process based on `config.py` settings.

**What This Section Does**:
- Reads `CITY_SELECTION_MODE` from config (ALL / ABOVE_1M / TEST / SPECIFIC)
- Lists all selected cities with their population data
- Displays available selection modes for reference

**Expected Output**: Ordered list of cities to process

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# City selection is controlled via config.py
# CITY_SELECTION_MODE options:
#   - "ALL"      : All 791 cities in the CSV file
#   - "ABOVE_1M" : 17 cities with population >= 1 million
#   - "TEST"     : 5 test cities (Lagos, Abuja, Kano, Jos, Port Harcourt)
#   - "SPECIFIC" : User-specified cities from any of the 791 valid cities

from aoi_utils import sort_cities_by_population

# Get cities to process from config, then sort by 2024 population (descending)
cities_to_process_raw = config.get_cities_to_process()
cities_to_process = sort_cities_by_population(cities_to_process_raw, cities_df, year='2024')

# Display configuration status
print(f"City Selection Configuration (from config.py)")
print(f"{'='*60}")
print(f"Current Mode: {config.CITY_SELECTION_MODE}")
print(f"{'='*60}")

# Show mode descriptions
mode_desc = {
    "ALL": f"All cities in CSV ({len(config.VALID_CITIES)} cities)",
    "ABOVE_1M": f"Cities >= 1M population ({len(config.CITIES_ABOVE_1M)} cities)",
    "TEST": f"Test cities ({len(config.TEST_CITIES)} cities: {', '.join(config.TEST_CITIES)})",
    "SPECIFIC": f"User-specified cities ({len(config.SELECTED_CITIES)} cities)"
}
print(f"Mode Description: {mode_desc.get(config.CITY_SELECTION_MODE, 'Unknown')}")

print(f"\nCities to process ({len(cities_to_process)}, sorted by 2024 population):")
for i, city in enumerate(cities_to_process, 1):
    pop_2024 = cities_df.loc[cities_df['Urban Agglomeration'] == city, '2024'].values
    pop_str = f"{pop_2024[0]:,.0f}k" if len(pop_2024) > 0 else "N/A"
    print(f"   {i:2d}. {city:<20s}  ({pop_str})")

# Show other modes info
print(f"\n{'='*60}")
print(f"Available modes (edit CITY_SELECTION_MODE in config.py):")
print(f"  'ALL'      - {len(config.VALID_CITIES)} cities (all in CSV)")
print(f"  'ABOVE_1M' - {len(config.CITIES_ABOVE_1M)} cities >= 1M population")
print(f"  'TEST'     - {len(config.TEST_CITIES)} test cities for quick runs")
print(f"  'SPECIFIC' - Custom selection from {len(config.VALID_CITIES)} valid cities")

<a id="process-aoi"></a>
## 4. Process City AOIs (Population-Ordered, Deduplicated)

**Purpose**: Generate validated AOI boundaries for every selected city, processing the most populated cities first and removing spatial duplicates.

**What This Section Does**:
- **Sorts** candidates by 2024 population (descending) so the largest cities claim their AOIs first
- Calls `process_city_aoi()` from `src/aoi_utils.py` for each candidate
- **Deduplicates**: after each AOI is created, all smaller candidate cities that fall inside it are marked as *absorbed* and **skipped** in subsequent iterations — their population is already counted in the parent AOI
- Applies configurable buffer distance around urban polygons
- Computes comprehensive metrics: area (km²), coverage efficiency, population within AOI

**Expected Output**: Reduced dictionary of unique, non-overlapping AOI results (fewer than the initial candidate count)

[↑ Back to Table of Contents](#Table-of-Contents)


In [ ]:
# Process all cities from config-based selection
# Cities are sorted by 2024 population (descending).
# Deduplication: if a city already falls inside a previously processed AOI,
# it is skipped — its population is already counted in the larger AOI.
print(f"Candidate cities: {len(cities_to_process)} (sorted by 2024 population)\n")

# Dictionary to store results for all cities
all_cities_results = {}

# Track which cities have already been absorbed into a larger AOI
absorbed_cities = {}          # city_name -> absorbing AOI name
cities_to_process_set = set(cities_to_process)

processed_count = 0

for idx, current_city in enumerate(cities_to_process, 1):
    # Skip cities already inside a previously processed AOI
    if current_city in absorbed_cities:
        print(f"\n⏭️  [{idx}/{len(cities_to_process)}] Skipping {current_city} "
              f"— already within '{absorbed_cities[current_city]}' AOI")
        continue

    processed_count += 1
    print(f"\n{'='*60}")
    print(f"Processing AOI {processed_count} [{idx}/{len(cities_to_process)}]: {current_city}")
    print(f"{'='*60}")

    # Process complete AOI using single function call
    aoi_results = process_city_aoi(
        current_city,
        cities_df,
        urban_gdf,
        boundary_gdf,
        buffer_distance=buffer_distance
    )

    # Store results
    all_cities_results[current_city] = aoi_results

    # Extract results for easier access
    city_info = aoi_results['city_info']
    aoi_gdf = aoi_results['aoi_gdf']
    cities_in_aoi_df = aoi_results['cities_in_aoi_df']
    comprehensive_stats = aoi_results['comprehensive_stats']
    metrics = aoi_results['metrics']

    # Mark every other candidate city inside this AOI as absorbed
    if not cities_in_aoi_df.empty:
        for contained_city in cities_in_aoi_df['Urban Agglomeration']:
            if contained_city != current_city and contained_city in cities_to_process_set:
                absorbed_cities[contained_city] = current_city

    # Display summary
    print(f"\nProcessing Summary for {current_city}:")
    print(f"AOI Area: {metrics['aoi_area_km2']:.2f} km²")
    print(f"Cities in AOI: {len(cities_in_aoi_df)}")
    print(f"Coverage Efficiency: {metrics['coverage_efficiency']:.1f}%")
    if len(cities_in_aoi_df) > 0:
        print(f"Total Population 2024: {comprehensive_stats['total_population_2024']:.0f}k")
        print(f"Major Cities: {comprehensive_stats['AOI_Major_Cities']}")

# Summary
print(f"\n{'='*60}")
print(f"DEDUPLICATION SUMMARY")
print(f"{'='*60}")
print(f"Candidate cities:  {len(cities_to_process)}")
print(f"Unique AOIs:       {len(all_cities_results)}")
print(f"Absorbed cities:   {len(absorbed_cities)}")
if absorbed_cities:
    print(f"\nAbsorbed city → Parent AOI:")
    for city, parent in absorbed_cities.items():
        print(f"  • {city} → {parent}")
print(f"{'='*60}")

<a id="export-data"></a>
## 5. Export Data

**Purpose**: Save all AOI results as structured files for downstream notebooks.

**What This Section Does**:
- Calls `export_all_city_data()` from `src/data_io.py`
- Exports GeoJSON boundaries (AOI, main polygon, adjacent polygons)
- Exports CSV tables (cities in AOI, population data)
- Exports comprehensive statistics as JSON
- Creates standardized folder structure: `City_[Name]/Features/`, `City_[Name]/Data/`

**Expected Output**: Complete file set per city in `data/outputs/01_aoi_delineation/`

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# Import export functions  
from data_io import export_all_city_data

# Export all data for each processed city
all_export_results = {}

for current_city, aoi_results in all_cities_results.items():
    print(f"\n{'='*60}")
    print(f"Exporting data for: {current_city}")
    print(f"{'='*60}")
    
    # Export all data for current city in the notebook-specific folder
    export_results = export_all_city_data(aoi_results, current_city, "01_aoi_delineation")
    all_export_results[current_city] = export_results
    
    # Display export summary
    print(f"Export Summary for {current_city}:")
    print(f"Output Directory: {export_results['output_directory']}")
    
    # Check which keys actually exist in the export_results
    print("Available keys:", list(export_results.keys()))
    
    # Try to access the directories with fallback
    features_dir = export_results.get('features_directory', 'Not available')
    figures_dir = export_results.get('figures_directory', 'Not available') 
    print(f"Features Directory: {features_dir}")
    print(f"Figures Directory: {figures_dir}")
    print(f"Total Files Exported: {export_results['total_files']}")
    
    # Show exported files by category
    for category, files in export_results['export_paths'].items():
        if isinstance(files, dict):
            print(f"\n{category.replace('_', ' ').title()}:")
            for name, path in files.items():
                filename = Path(path).name
                print(f"  - {name.replace('_', ' ')}: {filename}")
        elif isinstance(files, str):
            filename = Path(files).name
            print(f"{category.replace('_', ' ').title()}: {filename}")

print(f"\n{'='*60}")
print(f"Completed exporting data for all {len(all_cities_results)} cities!")
print(f"{'='*60}")


<a id="visualizations"></a>
## 6. Visualizations

<a id="viz-per-city"></a>
### 6a. Per-City Maps & Population Trends

**Purpose**: Create publication-ready maps and charts for each processed city.

**What This Section Does**:
- Generates static AOI maps using `create_aoi_map()` from `src/visualization.py`
- Creates interactive leaflet maps using `create_interactive_map()` from `src/visualization.py`
- Plots population trend projections using `create_population_trends_plot()` from `src/visualization.py`
- Saves all figures to `City_[Name]/Figures/` as SVG and SVG

**Expected Output**: Three visualizations per city (static map, interactive map, population trends)

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
# Import visualization functions
from visualization import create_aoi_map, create_interactive_map, create_population_trends_plot, save_figures

print("Creating visualizations for all cities...")

# Create visualizations for each processed city
all_figures_saved = {}

for current_city, aoi_results in all_cities_results.items():
    print(f"\n{'='*60}")
    print(f"Creating visualizations for: {current_city}")
    print(f"{'='*60}")
    
    # Create and display static AOI map
    print("1. Creating comprehensive AOI map...")
    aoi_fig = create_aoi_map(aoi_results, current_city)
    plt.show()
    
    # Create and display interactive map
    print("\n2. Creating interactive map...")
    interactive_map = create_interactive_map(aoi_results, current_city)
    display(interactive_map)
    
    # Create and display population trends plot
    print("\n3. Creating population trends plot...")
    pop_fig = create_population_trends_plot(aoi_results, current_city)
    if pop_fig:
        plt.show()
    
    # Save all figures to the Figures directory
    print("\n4. Saving all figures...")
    export_results = all_export_results[current_city]
    figures_saved = save_figures(aoi_results, current_city, export_results['figures_directory'])
    all_figures_saved[current_city] = figures_saved
    
    print(f"\nVisualization Summary:")
    print(f"Figures Directory: {export_results['figures_directory']}")
    for name, path in figures_saved.items():
        print(f"  - {name}: {Path(path).name}")

print(f"\n{'='*60}")
print(f"Completed visualizations for all {len(all_cities_results)} cities!")
print(f"{'='*60}")


<a id="viz-composite"></a>
### 6b. Composite Population Trend — All Nigerian Cities

**Purpose**: Aggregate population across **all selected cities** and plot the composite trend (1950–2035).

**What This Section Does**:
- Calls `create_composite_population_trend()` from `src/visualization.py`
- Plots total and top-5 city population curves with 2024 & 2035 annotations
- Adds linear regression with R² and 10-year growth rates
- Saves composite figure as SVG and SVG

| Element | Description |
|---------|-------------|
| **Total population curve** | Sum of all city populations per year |
| **Top-5 cities** | Individual lines for the 5 largest cities (by 2024 pop) |
| **2024 & 2035 markers** | Diamond markers with population annotations |
| **Linear regression** | Trend line with equation and R² annotation |
| **Growth rates** | 10-year growth rates for each decade |
| **Projection shading** | Grey shading for projected years (2025–2035) |

[↑ Back to Table of Contents](#Table-of-Contents)

In [ ]:
import importlib
import visualization
importlib.reload(visualization)
from visualization import create_composite_population_trend

# Output directory
save_dir = Path(config.PROJECT_ROOT) / 'data' / 'outputs' / '01_aoi_delineation'

# ┐-€ Helper to print composite stats ┐-€
def _print_composite_stats(stats, label):
    yr0 = stats.get('first_year', 1975)
    yr1 = stats.get('last_year', 2050)
    print(f"\n{chr(0x2500)*60}")
    print(f"{label}")
    print(f"AOIs/Cities included: {stats['n_aois']}")
    print(f"Population {yr0}: {stats['pop_first_k']:,.0f}k ({stats['pop_first_k']/1000:.1f} M)")
    print(f"Population 2024: {stats['pop_2024_k']:,.0f}k ({stats['pop_2024_k']/1000:.1f} M)")
    print(f"Population {yr1}: {stats['pop_last_k']:,.0f}k ({stats['pop_last_k']/1000:.1f} M)")
    print(f"Linear trend slope: {stats['slope_M_per_yr']:,.2f} M/yr")
    print(f"R\u00b2: {stats['r_squared']:.4f}")
    print(f"\u2713 Output folder: {save_dir}")

# ________________________________________________________________________________________
# 1) AOI-LEVEL COMPOSITE  (all cities within each AOI buffer)
# ________________________________________________________________________________________
print(f"\n{'='*80}")
print("COMPOSITE POPULATION TREND \u2014 AOI-LEVEL (all cities within each AOI buffer)")
print(f"{'='*80}\n")

fig_aoi, stats_aoi = create_composite_population_trend(
    all_cities_results, save_dir=str(save_dir), mode='aoi'
)
plt.show()
plt.close()
_print_composite_stats(stats_aoi, 'AOI-level summary')

# ________________________________________________________________________________________
# 2) INDIVIDUAL-CITY COMPOSITE  (67 cities >= 300k, no neighbours)
# ________________________________________________________________________________________
print(f"\n\n{'='*80}")
print("COMPOSITE POPULATION TREND \u2014 INDIVIDUAL CITIES (\u226567 cities \u2265300k)")
print(f"{'='*80}\n")

# cities_to_process_raw = original 67 cities from config (before spatial dedup)
fig_ind, stats_ind = create_composite_population_trend(
    all_cities_results, save_dir=str(save_dir), mode='individual',
    target_cities=cities_to_process_raw
)
plt.show()
plt.close()
_print_composite_stats(stats_ind, 'Individual-city summary')

print(f"\n{'='*80}")
print("Both composite plots saved successfully.")
print(f"{'='*80}")

<a id="summary"></a>
## 7. Summary & Next Steps

[↑ Back to Table of Contents](#Table-of-Contents)

---

### Key Accomplishments

This notebook successfully completed **AOI delineation** for all selected Nigerian cities:

#### ✅ Data Loading & Validation
- Loaded city coordinates, urban polygons, and national boundary
- Validated CRS compatibility and spatial integrity

#### ✅ AOI Processing
- Generated buffered urban AOI boundaries for each city
- Computed area (km²), coverage efficiency, and population metrics
- Identified co-located cities within buffer zones

#### ✅ Data Export
- Exported GeoJSON boundaries (AOI, main polygon, adjacent polygons)
- Saved population tables and comprehensive statistics
- Organized outputs in standardized `City_[Name]/` folder structure

#### ✅ Visualization
- Created static AOI maps, interactive leaflet maps, and population trend plots
- Generated composite population trend across all cities (1950–2035)
- Saved all figures as SVG (screen) and SVG (publication)

---

### Output Files Generated

All results saved to: `data/outputs/01_aoi_delineation/`

### Next Steps: Geospatial Feature Extraction (Notebook 01a)

The AOI boundaries are now ready for enrichment with geospatial features:

#### Notebook 01a: JRC Water Bodies + OSM Roads & Places

**What It Does**:
- Extracts JRC Global Surface Water bodies via Google Earth Engine
- Downloads OpenStreetMap roads and place names via Overpass API
- Creates publication-ready study area maps

**Why This Matters**:
- Water bodies are used to **mask water pixels** during LST processing
- Roads and place names provide **contextual overlays** on hotspot maps
- Publication map serves as the **study area figure** in manuscripts

---